# Mining Operations: Equipment & Safety Data Preparation

## Project Objective

Audit the equipment and safety tables of the mining operations database, document every data quality decision, and produce analysis-ready datasets for downstream analysis (equipment performance, safety reporting, and modelling).

## Scope

1. Load and audit the source tables (size, missing values, duplicates, keys, site IDs, date coverage).
2. Identify how each equipment metric is defined by testing candidate formulas, then validate the equipment data against rules.
3. Validate the safety incident and safety metrics tables against rules.
4. Record preparation decisions, and build the prepared equipment and safety datasets.
5. Export the datasets with an audit log and a data dictionary, and verify the files against the database.

## Data Source

Tables `equipment_performance`, `safety_incidents`, `safety_metrics_summary`, and `mine_sites` in `mining_operations.db`, read directly with SQL. Source values are never modified.

## Notebook Structure

| Section | Topic |
|---|---|
| 1 | Environment and database connection |
| 2 | Source checks |
| 3 | Equipment audit |
| 4 | Safety audit |
| 5 | Preparation decisions and prepared datasets |
| 6 | Export and verification |
| 7 | Summary |

## Output

Four files saved in a folder named `prepared_data` next to the database:

| File | Content |
|---|---|
| `equipment_prepared.csv` | Equipment records (one row per unit per day) with added flags and calendar columns |
| `safety_prepared.csv` | Safety incidents (one row per incident) with added flags and calendar columns |
| `data_quality_audit.csv` | Every audit rule, its violation count, and its status |
| `prepared_data_dictionary.csv` | Source and added columns of both datasets, with descriptions |

> **Important note:** The dataset is synthetic and intended for portfolio demonstration. Findings describe the data, not a real mining company.

## 1. Environment and Database Connection

In [1]:
import os
import sqlite3
from pathlib import Path

import numpy as np
import pandas as pd
from google.colab import drive

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 200)

# Mount Google Drive (safe to re-run)
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
# Locate the SQLite database in Google Drive
db_candidates = [
    Path(root) / file
    for root, dirs, files in os.walk('/content/drive/MyDrive')
    for file in files
    if file.lower() == 'mining_operations.db'
]

print("Database files found:", len(db_candidates))
for path in db_candidates:
    print(path)

if not db_candidates:
    raise ValueError("Database not found. Check its location in Google Drive.")

if len(db_candidates) > 1:
    print("\nWarning: more than one file found. The first one is used.")

DB_PATH = db_candidates[0]

# All exported files go into one dedicated folder next to the database
OUTPUT_DIR = DB_PATH.parent / "prepared_data"
OUTPUT_DIR.mkdir(exist_ok=True)

conn = sqlite3.connect(DB_PATH)

print("\nDatabase connection successful!")
print("Database path:", DB_PATH)
print("Output folder:", OUTPUT_DIR)

tables = pd.read_sql_query(
    "SELECT name AS table_name FROM sqlite_master WHERE type = 'table' ORDER BY name", conn
)
print("Tables in the database:", ", ".join(tables["table_name"]))

Database files found: 1
/content/drive/MyDrive/Mining-Operation/mining_operations.db

Database connection successful!
Database path: /content/drive/MyDrive/Mining-Operation/mining_operations.db
Output folder: /content/drive/MyDrive/Mining-Operation/prepared_data
Tables in the database: daily_costs, daily_production, equipment_performance, mine_sites, safety_incidents, safety_metrics_summary


In [3]:
# Audit log: every rule checked in this notebook is added here and exported at the end
audit_log = []


def record_rule(area, rule, violating_rows, note=""):
    # A rule passes when it has no violating rows
    audit_log.append({
        "area": area,
        "check": rule,
        "violating_rows": int(violating_rows),
        "status": "Pass" if int(violating_rows) == 0 else "Review",
        "note": note,
    })


def show_rules(area):
    table = pd.DataFrame([r for r in audit_log if r["area"] == area])
    display(table[["check", "violating_rows", "status"]])
    review = table[table["status"] == "Review"]
    if review.empty:
        print(f"All {len(table)} {area.lower()} rules pass.")
    else:
        print(f"{len(review)} rule(s) need review:", "; ".join(review["check"]))

## 2. Source Checks

Ukuran, nilai kosong, duplikat, keunikan key, dan kecocokan `site_id` untuk setiap tabel sumber.

In [4]:
equipment_raw = pd.read_sql_query("SELECT * FROM equipment_performance", conn)
safety_raw = pd.read_sql_query("SELECT * FROM safety_incidents", conn)
metrics_raw = pd.read_sql_query("SELECT * FROM safety_metrics_summary", conn)
sites_df = pd.read_sql_query("SELECT * FROM mine_sites", conn)

source_tables = [
    ("equipment_performance", equipment_raw, ["date", "equipment_id"]),
    ("safety_incidents", safety_raw, ["incident_id"]),
    ("safety_metrics_summary", metrics_raw, ["site_id"]),
    ("mine_sites", sites_df, ["site_id"]),
]

rows = []
for table_name, frame, key_columns in source_tables:
    rows.append({
        "table_name": table_name,
        "rows": len(frame),
        "columns": frame.shape[1],
        "missing_values": int(frame.isna().sum().sum()),
        "missing_keys": int(frame[key_columns].isna().any(axis=1).sum()),
        "duplicate_rows": int(frame.duplicated().sum()),
        "duplicate_keys": int(frame.duplicated(key_columns).sum()),
    })
    record_rule("Source", f"{table_name}: duplicate rows", frame.duplicated().sum())
    record_rule("Source", f"{table_name}: missing or duplicate keys",
                frame[key_columns].isna().any(axis=1).sum() + frame.duplicated(key_columns).sum())

source_checks = pd.DataFrame(rows)
display(source_checks)

for table_name, frame, _ in source_tables:
    missing = frame.isna().sum()
    missing = missing[missing > 0]
    if len(missing):
        print(f"{table_name}: columns with missing values -> {missing.to_dict()}")

,table_name,rows,columns,missing_values,missing_keys,duplicate_rows,duplicate_keys
0,equipment_performance,8030,16,0,0,0,0
1,safety_incidents,360,11,0,0,0,0
2,safety_metrics_summary,4,7,0,0,0,0
3,mine_sites,5,16,1,0,0,0


mine_sites: columns with missing values -> {'automation_level': 1}


In [5]:
# Site IDs used in each table must exist in mine_sites
known_sites = set(sites_df["site_id"])
site_rows = []
for table_name, frame in [("equipment_performance", equipment_raw), ("safety_incidents", safety_raw),
                          ("safety_metrics_summary", metrics_raw)]:
    used = set(frame["site_id"])
    unknown = used - known_sites
    site_rows.append({
        "table_name": table_name,
        "sites_in_table": len(used),
        "sites_in_mine_sites": len(known_sites),
        "unknown_site_ids": len(unknown),
        "mine_sites_without_records": ", ".join(sorted(known_sites - used)) or "none",
    })
    record_rule("Source", f"{table_name}: site IDs not found in mine_sites", len(unknown))

site_coverage = pd.DataFrame(site_rows)
display(site_coverage)

# Status of the site(s) that have no equipment or safety records
display(
    sites_df.loc[~sites_df["site_id"].isin(set(equipment_raw["site_id"])),
                 ["site_id", "site_name", "commodity", "status", "workforce_count", "automation_level"]]
)

,table_name,sites_in_table,sites_in_mine_sites,unknown_site_ids,mine_sites_without_records
0,equipment_performance,4,5,0,NICK001
1,safety_incidents,4,5,0,NICK001
2,safety_metrics_summary,4,5,0,NICK001


,site_id,site_name,commodity,status,workforce_count,automation_level
4,NICK001,Nickel Point Mine,Nickel,Care & Maintenance,25,NaN


**Interpretasi**

- Tidak ada baris duplikat, key kosong, maupun key ganda pada keempat tabel, dan semua `site_id` dikenali oleh `mine_sites`. Satu-satunya nilai kosong adalah `mine_sites.automation_level` milik Nickel Point, yang tidak dipakai di notebook ini.
- `equipment_performance` berisi **8.030 baris** (22 unit × 365 hari), `safety_incidents` berisi **360 insiden**, dan `safety_metrics_summary` berisi 4 baris (satu per lokasi).
- **Nickel Point** (*Care & Maintenance*, 25 pekerja) tidak memiliki data peralatan, insiden, maupun ringkasan keselamatan. Ini sesuai statusnya, sehingga dicatat sebagai temuan audit dan **tidak** diisi dengan nol.

## 3. Equipment Audit

### 3.1 Which formula defines each stored metric?

Persentase dan jam peralatan dihitung ulang dari kolom jam. Beberapa rumus kandidat diuji untuk `availability_pct` dan `utilization_pct`, karena rumus yang dipakai menentukan apakah data tampak konsisten atau salah. Kandidat yang salah akan tampak sebagai ribuan "ketidakkonsistenan" yang sebenarnya hanya akibat rumus uji yang keliru.

In [6]:
equipment_df = equipment_raw.copy()
equipment_df["date"] = pd.to_datetime(equipment_df["date"], errors="coerce")
equipment_df = equipment_df.merge(sites_df[["site_id", "site_name"]], on="site_id", how="left")

available = equipment_df["available_hours"].where(equipment_df["available_hours"] > 0)

candidates = {
    "availability_pct": {
        "available hours / 24": equipment_df["available_hours"] / 24 * 100,
        "operating / available hours": equipment_df["operating_hours"] / available * 100,
        "(operating + idle) / available hours": (equipment_df["operating_hours"] + equipment_df["idle_hours"]) / available * 100,
        "(available - maintenance) / available hours": (equipment_df["available_hours"] - equipment_df["maintenance_hours"]) / available * 100,
        "(available - breakdown) / available hours": (equipment_df["available_hours"] - equipment_df["breakdown_hours"]) / available * 100,
    },
    "utilization_pct": {
        "operating / available hours": equipment_df["operating_hours"] / available * 100,
        "(operating + idle) / available hours": (equipment_df["operating_hours"] + equipment_df["idle_hours"]) / available * 100,
        "operating / 24": equipment_df["operating_hours"] / 24 * 100,
    },
    "oee_pct": {
        "availability x utilization": equipment_df["availability_pct"] * equipment_df["utilization_pct"] / 100,
        "utilization only": equipment_df["utilization_pct"],
    },
}

formula_rows = []
for stored_column, options in candidates.items():
    for label, values in options.items():
        gap = (equipment_df[stored_column] - values).abs()
        formula_rows.append({
            "stored_column": stored_column,
            "tested_formula": label,
            "rows_over_1_point": int((gap > 1.0).sum()),
            "mean_difference_pts": round(float(gap.mean()), 3),
            "max_difference_pts": round(float(gap.max()), 2),
        })

formula_comparison = pd.DataFrame(formula_rows)
display(formula_comparison)

best_formula = (
    formula_comparison.sort_values(["stored_column", "rows_over_1_point", "mean_difference_pts"])
    .groupby("stored_column").first()[["tested_formula", "rows_over_1_point", "max_difference_pts"]]
)
print("Formula that reproduces each stored column:")
display(best_formula)

,stored_column,tested_formula,rows_over_1_point,mean_difference_pts,max_difference_pts
0,availability_pct,available hours / 24,0,0.009,0.25
1,availability_pct,operating / available hours,7618,21.738,50.00
2,availability_pct,(operating + idle) / available hours,651,1.671,59.61
3,availability_pct,(available - maintenance) / available hours,620,0.842,49.50
4,availability_pct,(available - breakdown) / available hours,650,1.391,54.53
5,utilization_pct,operating / available hours,0,0.098,0.69
6,utilization_pct,(operating + idle) / available hours,7608,22.528,50.70
7,utilization_pct,operating / 24,651,1.375,49.58
8,oee_pct,availability x utilization,0,0.003,0.12
9,oee_pct,utilization only,651,1.290,49.50


Formula that reproduces each stored column:


,tested_formula,rows_over_1_point,max_difference_pts
stored_column,,,
availability_pct,available hours / 24,0,0.25
oee_pct,availability x utilization,0,0.12
utilization_pct,operating / available hours,0,0.69


**Interpretasi rumus**

| Kolom | Rumus yang cocok | Baris berbeda > 1 poin | Selisih maksimum |
|---|---|---|---|
| `availability_pct` | `available_hours` ÷ 24 | 0 | 0,25 poin |
| `utilization_pct` | `operating_hours` ÷ `available_hours` | 0 | 0,69 poin |
| `oee_pct` | availability × utilization | 0 | 0,12 poin |

- Ketiga persentase konsisten dengan jam pada seluruh 8.030 baris bila memakai definisi yang benar. Selisih kecil berasal dari pembulatan jam ke 0,1.
- **Availability** di dataset ini berarti jam yang tersedia dari 24 jam (setelah dikurangi maintenance dan breakdown), **bukan** proporsi jam operasi. Rumus `operating ÷ available` menyimpang pada 7.618 baris (rata-rata 21,7 poin). Itu kesalahan rumus uji, bukan kesalahan data.
- Rumus yang tampak "hampir benar" perlu diwaspadai: `(available − maintenance) ÷ available` hanya berselisih rata-rata 0,84 poin, tetapi tetap menyimpang pada 620 baris (hingga 49,5 poin). Rumus itu cocok hanya pada hari tanpa breakdown, sehingga rata-rata yang kecil menyembunyikan baris yang salah. Rumus yang benar tidak memiliki satu pun baris menyimpang.
- Sebanyak 651 baris menyimpang pada beberapa rumus kandidat. Jumlah itu sama dengan jumlah hari yang memiliki maintenance atau breakdown (Section 3.3).

### 3.2 Rule-based checks

Setiap aturan di bawah ini seharusnya menghasilkan **0 baris pelanggaran**. Hubungan jam yang dipakai: `operating + idle = available` dan `available = 24 − maintenance − breakdown`.

In [7]:
hour_columns = ["available_hours", "operating_hours", "idle_hours", "maintenance_hours", "breakdown_hours"]
percent_columns = ["availability_pct", "utilization_pct", "oee_pct"]
HOUR_TOLERANCE = 0.15  # hours are stored with one decimal, so +/-0.1 is rounding

e = equipment_df
units_with_other_coverage = (
    e.groupby("equipment_id")["date"].nunique() != 365
).sum()
type_conflicts = (
    e.groupby("equipment_id")[["equipment_type", "model", "autonomous"]].nunique() > 1
).any(axis=1).sum()

equipment_rules = [
    ("Missing values in any column", e.isna().sum().sum()),
    ("Invalid dates", e["date"].isna().sum()),
    ("Negative hours, fuel, or maintenance cost",
     (e[hour_columns + ["fuel_consumed_liters", "maintenance_cost_aud"]] < 0).any(axis=1).sum()),
    ("Availability, utilization, or OEE outside 0-100", (~e[percent_columns].apply(lambda c: c.between(0, 100))).any(axis=1).sum()),
    ("Available hours above 24", (e["available_hours"] > 24).sum()),
    ("Available hours equal to 0 (utilization undefined)", (e["available_hours"] <= 0).sum()),
    ("Operating + idle hours differ from available hours (> 0.15 h)",
     ((e["operating_hours"] + e["idle_hours"] - e["available_hours"]).abs() > HOUR_TOLERANCE).sum()),
    ("Available hours differ from 24 - maintenance - breakdown (> 0.15 h)",
     ((e["available_hours"] - (24 - e["maintenance_hours"] - e["breakdown_hours"])).abs() > HOUR_TOLERANCE).sum()),
    ("Operating + idle + maintenance + breakdown differ from 24 hours (> 0.15 h)",
     ((e[hour_columns[1:]].sum(axis=1) - 24).abs() > HOUR_TOLERANCE).sum()),
    ("availability_pct differs from available hours / 24 (> 1 pt)",
     ((e["availability_pct"] - e["available_hours"] / 24 * 100).abs() > 1).sum()),
    ("utilization_pct differs from operating / available hours (> 1 pt)",
     ((e["utilization_pct"] - e["operating_hours"] / available * 100).abs() > 1).sum()),
    ("OEE differs from availability x utilization (> 0.5 pts)",
     ((e["oee_pct"] - e["availability_pct"] * e["utilization_pct"] / 100).abs() > 0.5).sum()),
    ("equipment_id with more than one type, model, or operation mode", type_conflicts),
    ("Units without exactly 365 distinct days of data", units_with_other_coverage),
]
for rule, count in equipment_rules:
    record_rule("Equipment", rule, count)

show_rules("Equipment")

,check,violating_rows,status
0,Missing values in any column,0,Pass
1,Invalid dates,0,Pass
2,"Negative hours, fuel, or maintenance cost",0,Pass
3,"Availability, utilization, or OEE outside 0-100",0,Pass
4,Available hours above 24,0,Pass
5,Available hours equal to 0 (utilization undefi...,0,Pass
6,Operating + idle hours differ from available h...,0,Pass
7,Available hours differ from 24 - maintenance -...,0,Pass
8,Operating + idle + maintenance + breakdown dif...,0,Pass
9,availability_pct differs from available hours ...,0,Pass


All 14 equipment rules pass.


**Interpretasi aturan peralatan**

- Seluruh **14 aturan lolos**: tidak ada nilai kosong, negatif, atau di luar rentang, dan setiap unit memiliki tepat 365 hari data.
- Hubungan jam yang berlaku: `operating_hours` + `idle_hours` = `available_hours`, dan `available_hours` = 24 − `maintenance_hours` − `breakdown_hours`. Karena itu jumlah **keempat** jenis jam (operating, idle, maintenance, breakdown) selalu 24 jam.
- Akibatnya, membandingkan jumlah keempat jam itu dengan `available_hours` akan selalu tampak "melebihi" pada hari yang memiliki maintenance atau breakdown. Itu bukan masalah data, dan karena itu tidak ada kolom penanda "jam melebihi available" pada dataset siap analisis.

### 3.3 Rounding differences, maintenance and breakdown

In [8]:
# Hour balance after rounding: how large are the differences between operating + idle and available hours?
hour_difference = (e["operating_hours"] + e["idle_hours"] - e["available_hours"]).round(2)
rounding_summary = (
    hour_difference.value_counts().sort_index().rename_axis("difference_hours").reset_index(name="rows")
)
rounding_summary["share_pct"] = (rounding_summary["rows"] / len(e) * 100).round(2)
display(rounding_summary)

# Maintenance and breakdown by day type
e = e.assign(
    has_breakdown=e["breakdown_hours"] > 0,
    has_maintenance=e["maintenance_hours"] > 0,
)
e["day_type"] = np.select(
    [e["has_breakdown"] & e["has_maintenance"], e["has_breakdown"], e["has_maintenance"]],
    ["Breakdown and maintenance", "Breakdown only", "Maintenance only"],
    default="Normal day",
)
day_type_summary = (
    e.groupby("day_type")
    .agg(
        rows=("equipment_id", "count"),
        avg_available_hours=("available_hours", "mean"),
        avg_oee_pct=("oee_pct", "mean"),
        avg_maintenance_hours=("maintenance_hours", "mean"),
        avg_breakdown_hours=("breakdown_hours", "mean"),
        avg_maintenance_cost_aud=("maintenance_cost_aud", "mean"),
    )
    .round(2)
    .reindex(["Normal day", "Maintenance only", "Breakdown only", "Breakdown and maintenance"])
    .reset_index()
)
day_type_summary["share_pct"] = (day_type_summary["rows"] / len(e) * 100).round(2)
display(day_type_summary)

print("Shortest positive breakdown (hours):", e.loc[e["has_breakdown"], "breakdown_hours"].min())
print("Longest breakdown (hours):", e["breakdown_hours"].max())
print("Rows where maintenance + breakdown exceed 24 hours:",
      int(((e["maintenance_hours"] + e["breakdown_hours"]) > 24).sum()))

,difference_hours,rows,share_pct
0,-0.1,68,0.85
1,0.0,7870,98.01
2,0.1,92,1.15


,day_type,rows,avg_available_hours,avg_oee_pct,avg_maintenance_hours,avg_breakdown_hours,avg_maintenance_cost_aud,share_pct
0,Normal day,7379,24.00,77.51,0.00,0.00,1249.74,91.89
1,Maintenance only,498,20.01,64.19,3.99,0.00,10018.87,6.20
2,Breakdown only,145,16.10,51.68,0.00,7.90,49667.16,1.81
3,Breakdown and maintenance,8,13.35,40.62,3.40,7.24,10344.98,0.10


Shortest positive breakdown (hours): 4.0
Longest breakdown (hours): 11.9
Rows where maintenance + breakdown exceed 24 hours: 0


**Interpretasi**

- **Pembulatan:** 7.870 baris (98,0%) memiliki operating + idle persis sama dengan available, dan 160 baris (2,0%) berbeda tepat 0,1 jam (68 baris −0,1 dan 92 baris +0,1). Itu pembulatan satu desimal. Nilai tidak diubah, hanya ditandai dengan `hours_rounding_flag`.
- **Breakdown jarang tetapi berat:** hanya 153 baris (1,9%) yang memiliki breakdown, terdiri atas 145 baris breakdown saja dan 8 baris breakdown sekaligus maintenance. Breakdown terpendek 4,0 jam dan terpanjang 11,9 jam, jadi tidak ada breakdown singkat.
- **OEE turun pada hari downtime:** hari normal (91,9% baris) memiliki OEE rata-rata 77,5%, hari maintenance saja 64,2%, dan hari breakdown sekitar 41% hingga 52%. Biaya perawatan rata-rata sekitar A$1.250 per hari normal, A$10.019 per hari maintenance saja, dan A$49.667 per hari breakdown saja.
- **8 hari dengan maintenance dan breakdown sekaligus** dipertahankan dan ditandai dengan `maintenance_breakdown_overlap_flag`. Contohnya terlalu sedikit untuk disimpulkan lebih jauh. Tidak ada hari dengan maintenance + breakdown di atas 24 jam.

### 3.4 Equipment inventory

In [9]:
inventory = (
    equipment_df.assign(operation_mode=np.where(equipment_df["autonomous"] == 1, "Autonomous", "Conventional"))
    .groupby(["site_name", "equipment_type", "model", "operation_mode"])
    .agg(units=("equipment_id", "nunique"), records=("equipment_id", "count"))
    .reset_index()
)
display(inventory)
print("Units:", equipment_df["equipment_id"].nunique(), "| Records:", len(equipment_df),
      "| Date range:", equipment_df["date"].min().date(), "to", equipment_df["date"].max().date())

,site_name,equipment_type,model,operation_mode,units,records
0,Copper Hills Mine,Jumbo Drill,Sandvik DD421,Conventional,1,365
1,Copper Hills Mine,Jumbo Drill,Sandvik DD422i,Conventional,1,365
2,Copper Hills Mine,LHD Loader,CAT R1700,Conventional,2,730
3,Copper Hills Mine,LHD Loader,Sandvik LH517i,Conventional,1,365
4,Golden Valley Mine,Dozer,CAT D10T2,Conventional,1,365
5,Golden Valley Mine,Dozer,CAT D11T,Conventional,1,365
6,Golden Valley Mine,Excavator,CAT 6060,Conventional,1,365
7,Golden Valley Mine,Excavator,Liebherr R9800,Conventional,1,365
8,Golden Valley Mine,Haul Truck,CAT 793F,Conventional,3,1095
9,Iron Ridge Mine,Dozer,CAT D11T,Conventional,1,365


Units: 22 | Records: 8030 | Date range: 2025-01-01 to 2025-12-31


## 4. Safety Audit

### 4.1 Incident records

In [10]:
safety_df = safety_raw.copy()
safety_df["date"] = pd.to_datetime(safety_df["date"], errors="coerce")
s = safety_df

lti = s["incident_type"].str.contains(r"\(LTI\)", regex=True)
mti = s["incident_type"].str.contains(r"\(MTI\)", regex=True)

workforce_by_site = sites_df.set_index("site_id")["workforce_count"]

safety_rules = [
    ("Missing values in any column", s.isna().sum().sum()),
    ("Invalid dates or dates outside 2025", (s["date"].isna() | (s["date"].dt.year != 2025)).sum()),
    ("Incident IDs not matching INC-nnnnn", (~s["incident_id"].str.match(r"^INC-\d{5}$")).sum()),
    ("Negative days_lost", (s["days_lost"] < 0).sum()),
    ("days_lost above 0 on a non-LTI incident", ((s["days_lost"] > 0) & ~lti).sum()),
    ("LTI with zero days_lost", (lti & (s["days_lost"] == 0)).sum()),
    ("High severity that is not an LTI (or an LTI that is not High)", ((s["severity"] == "High") != lti).sum()),
    ("Unknown severity value", (~s["severity"].isin(["Low", "Medium", "High"])).sum()),
    ("workforce_size differs from mine_sites.workforce_count", (s["workforce_size"] != s["site_id"].map(workforce_by_site)).sum()),
    ("site_name differs from mine_sites", (s["site_name"] != s["site_id"].map(sites_df.set_index("site_id")["site_name"])).sum()),
]
for rule, count in safety_rules:
    record_rule("Safety", rule, count)

show_rules("Safety")

,check,violating_rows,status
0,Missing values in any column,0,Pass
1,Invalid dates or dates outside 2025,0,Pass
2,Incident IDs not matching INC-nnnnn,0,Pass
3,Negative days_lost,0,Pass
4,days_lost above 0 on a non-LTI incident,0,Pass
5,LTI with zero days_lost,0,Pass
6,High severity that is not an LTI (or an LTI th...,0,Pass
7,Unknown severity value,0,Pass
8,workforce_size differs from mine_sites.workfor...,0,Pass
9,site_name differs from mine_sites,0,Pass


All 10 safety rules pass.


In [11]:
type_severity = (
    s.groupby(["incident_type", "severity"])
    .agg(incidents=("incident_id", "count"), total_days_lost=("days_lost", "sum"),
         max_days_lost=("days_lost", "max"), open_investigations=("investigation_status", lambda x: int((x != "Closed").sum())))
    .reset_index()
    .sort_values("incidents", ascending=False)
)
display(type_severity)

by_site = (
    s.pivot_table(index="site_name", columns="incident_type", values="incident_id", aggfunc="count", fill_value=0)
)
by_site["total"] = by_site.sum(axis=1)
display(by_site)

print("Investigation status:")
display(s["investigation_status"].value_counts().rename_axis("investigation_status").reset_index(name="incidents"))
print("Root causes:")
display(s["root_cause"].value_counts().rename_axis("root_cause").reset_index(name="incidents"))

,incident_type,severity,incidents,total_days_lost,max_days_lost,open_investigations
4,Near Miss,Low,160,0,0,42
1,First Aid Injury (FAI),Low,80,0,0,23
5,Potential Significant Incident (PSI),Medium,60,0,0,17
3,Medical Treatment Injury (MTI),Medium,32,0,0,9
0,Environmental Incident,Medium,20,0,0,4
2,Lost Time Injury (LTI),High,8,111,28,2


incident_type,Environmental Incident,First Aid Injury (FAI),Lost Time Injury (LTI),Medical Treatment Injury (MTI),Near Miss,Potential Significant Incident (PSI),total
site_name,,,,,,,
Copper Hills Mine,5,20,2,8,40,15,90
Golden Valley Mine,5,20,2,8,40,15,90
Iron Ridge Mine,5,20,2,8,40,15,90
Lithium Creek Mine,5,20,2,8,40,15,90


Investigation status:


,investigation_status,incidents
0,Closed,263
1,Under Investigation,97


Root causes:


,root_cause,incidents
0,Human Error,78
1,Procedural Gap,76
2,Training Gap,74
3,Environmental Factor,72
4,Equipment Failure,60


**Interpretasi aturan keselamatan**

- Seluruh **10 aturan lolos**. Hari hilang hanya muncul pada insiden LTI, dan seluruh insiden *High* adalah LTI (8 insiden, 111 hari hilang, paling lama 28 hari). Insiden *Low* dan *Medium* tidak mencatat hari hilang.
- **Setiap lokasi memiliki tepat 90 insiden dengan komposisi yang sama**: 2 LTI, 8 MTI, 20 FAI, 15 PSI, 40 near miss, dan 5 insiden lingkungan. Ini ciri data sintetis. Akibatnya, perbedaan TRIFR dan LTIFR antar lokasi (Section 4.3) murni berasal dari jam kerja, bukan dari jumlah insiden.
- Akar masalah tersebar merata (60 hingga 78 insiden per kategori: *Human Error* 78, *Procedural Gap* 76, *Training Gap* 74, *Environmental Factor* 72, *Equipment Failure* 60), tanpa satu penyebab yang dominan.
- Sebanyak **97 insiden (26,9%)** masih berstatus *Under Investigation*.

### 4.2 Open investigations

Insiden yang belum ditutup adalah hal yang bisa ditindaklanjuti dari data keselamatan, terutama bila menyangkut cedera hari hilang (LTI).

In [12]:
open_cases = s[s["investigation_status"] != "Closed"].copy()
open_cases["days_open_at_year_end"] = (pd.Timestamp("2025-12-31") - open_cases["date"]).dt.days

open_by_severity = (
    s.assign(is_open=s["investigation_status"] != "Closed")
    .groupby("severity")
    .agg(incidents=("incident_id", "count"), open_investigations=("is_open", "sum"))
    .assign(open_share_pct=lambda t: (t["open_investigations"] / t["incidents"] * 100).round(1))
    .reindex(["High", "Medium", "Low"])
    .reset_index()
)
display(open_by_severity)

print("Open investigations of lost time injuries (High severity):")
display(
    open_cases[open_cases["severity"] == "High"]
    [["incident_id", "date", "site_name", "description", "days_lost", "root_cause", "investigation_status", "days_open_at_year_end"]]
    .sort_values("date")
)

print("Open investigations, age at 31 Dec 2025 (days):")
display(open_cases["days_open_at_year_end"].describe().round(1).to_frame("days_open"))

,severity,incidents,open_investigations,open_share_pct
0,High,8,2,25.0
1,Medium,112,30,26.8
2,Low,240,65,27.1


Open investigations of lost time injuries (High severity):


,incident_id,date,site_name,description,days_lost,root_cause,investigation_status,days_open_at_year_end
90,INC-00091,2025-03-21,Golden Valley Mine,Crush injury,10,Equipment Failure,Under Investigation,285
91,INC-00092,2025-11-14,Golden Valley Mine,Fall from height,10,Procedural Gap,Under Investigation,47


Open investigations, age at 31 Dec 2025 (days):


,days_open
count,97.0
mean,177.9
std,101.0
min,8.0
25%,108.0
50%,160.0
75%,267.0
max,364.0


**Interpretasi**

- Persentase investigasi terbuka hampir sama di semua tingkat keparahan (25,0% *High*, 26,8% *Medium*, 27,1% *Low*). Penutupan investigasi tidak tampak diprioritaskan menurut keparahan.
- **Dua LTI belum ditutup, keduanya di Golden Valley**: INC-00091 (*crush injury*, 21 Maret, 10 hari hilang, terbuka sekitar 285 hari pada akhir tahun) dan INC-00092 (*fall from height*, 14 November, 10 hari hilang, terbuka sekitar 47 hari).
- Dari 97 investigasi terbuka, rata-rata usianya sekitar 178 hari (median 160 hari, tertua 364 hari). Usia dihitung sampai 31 Desember 2025 karena data hanya mencakup 2025 dan tidak memuat tanggal penutupan, jadi angkanya batas bawah usia sesungguhnya bila investigasi masih berjalan.

### 4.3 Safety metrics summary

In [13]:
m = metrics_raw.copy()
m["calculated_trifr"] = (m["lti_count"] + m["mti_count"]) * 1_000_000 / m["total_hours_worked"]
m["calculated_ltifr"] = m["lti_count"] * 1_000_000 / m["total_hours_worked"]
m["workforce_count"] = m["site_id"].map(workforce_by_site)
m["hours_per_worker"] = m["total_hours_worked"] / m["workforce_count"]
m["lti_in_incidents"] = m["site_id"].map(s[lti].groupby("site_id").size()).fillna(0).astype(int)
m["mti_in_incidents"] = m["site_id"].map(s[mti].groupby("site_id").size()).fillna(0).astype(int)

display(
    m[["site_id", "total_hours_worked", "workforce_count", "hours_per_worker", "lti_count", "lti_in_incidents",
       "mti_count", "mti_in_incidents", "trifr", "calculated_trifr", "ltifr", "calculated_ltifr"]].round(2)
)

metrics_rules = [
    ("TRIFR differs from (LTI + MTI) x 1,000,000 / hours (> 0.01)", ((m["trifr"] - m["calculated_trifr"]).abs() > 0.01).sum()),
    ("LTIFR differs from LTI x 1,000,000 / hours (> 0.01)", ((m["ltifr"] - m["calculated_ltifr"]).abs() > 0.01).sum()),
    ("LTI count differs from safety_incidents", (m["lti_count"] != m["lti_in_incidents"]).sum()),
    ("MTI count differs from safety_incidents", (m["mti_count"] != m["mti_in_incidents"]).sum()),
    ("Sites with incidents but no row in safety_metrics_summary", len(set(s["site_id"]) - set(m["site_id"]))),
]
for rule, count in metrics_rules:
    record_rule("Safety metrics", rule, count)

show_rules("Safety metrics")
print("Hours worked per worker (distinct values):", sorted(m["hours_per_worker"].round(1).unique()))

,site_id,total_hours_worked,workforce_count,hours_per_worker,lti_count,lti_in_incidents,mti_count,mti_in_incidents,trifr,calculated_trifr,ltifr,calculated_ltifr
0,IRON001,3723000,850,4380.0,2,2,8,8,2.69,2.69,0.54,0.54
1,GOLD001,5256000,1200,4380.0,2,2,8,8,1.90,1.90,0.38,0.38
2,LITH001,1971000,450,4380.0,2,2,8,8,5.07,5.07,1.01,1.01
3,COPP001,1226400,280,4380.0,2,2,8,8,8.15,8.15,1.63,1.63


,check,violating_rows,status
0,"TRIFR differs from (LTI + MTI) x 1,000,000 / h...",0,Pass
1,"LTIFR differs from LTI x 1,000,000 / hours (> ...",0,Pass
2,LTI count differs from safety_incidents,0,Pass
3,MTI count differs from safety_incidents,0,Pass
4,Sites with incidents but no row in safety_metr...,0,Pass


All 5 safety metrics rules pass.
Hours worked per worker (distinct values): [np.float64(4380.0)]


**Interpretasi**

- Seluruh **5 aturan lolos**. TRIFR dan LTIFR yang tersimpan sama dengan hasil hitung ulang dari jam kerja, dan jumlah LTI dan MTI pada tabel ringkasan sama dengan catatan insiden.
- Jam kerja per pekerja sama di semua lokasi, yaitu **4.380 jam** (12 jam × 365 hari). Jadi `total_hours_worked` adalah hasil perkalian jumlah pekerja × 4.380, bukan data absensi sesungguhnya.
- Karena jumlah insiden identik di setiap lokasi, rate hanya berbeda karena jumlah pekerja. Copper Hills (280 pekerja) memiliki TRIFR tertinggi (8,15) dan Golden Valley (1.200 pekerja) terendah (1,90). Rate yang lebih tinggi di lokasi kecil mencerminkan penyebut yang kecil, bukan lokasi yang lebih berbahaya.
- Nickel Point tidak tercakup, sesuai statusnya (Section 2).

## 5. Preparation Decisions and Prepared Datasets

Prinsip: **nilai sumber tidak diubah, diisi, atau dihapus**. Persiapan hanya menambahkan kolom penanda dan kolom kalender, sehingga setiap baris tetap dapat ditelusuri ke tabel sumbernya.

| Temuan audit | Keputusan |
|---|---|
| Selisih ±0,1 jam antara operating + idle dan available | Dipertahankan. Ditandai dengan `hours_rounding_flag` (pembulatan satu desimal). |
| Hari dengan jam maintenance dan breakdown sekaligus | Dipertahankan. Ditandai dengan `maintenance_breakdown_overlap_flag`. |
| Jam maintenance dan breakdown | Sudah dikurangkan dari `available_hours`, sehingga tidak ada kolom "jam melebihi available". Total downtime dihitung sebagai `downtime_hours`. |
| Availability, utilization, OEE | Rumus sudah terkonfirmasi (Section 3.1). Kolom tersimpan dipakai apa adanya. |
| Nickel Point tanpa data peralatan dan keselamatan | Dipertahankan sebagai catatan audit. Tidak diisi nol. |
| Insiden yang belum ditutup | Ditandai dengan `is_open_investigation`. |

In [14]:
# ---- Equipment: source columns first, then added columns
source_equipment_columns = list(equipment_raw.columns)

equipment_prepared = equipment_df.copy()
equipment_prepared["operation_mode"] = np.where(equipment_prepared["autonomous"] == 1, "Autonomous", "Conventional")
equipment_prepared["month"] = equipment_prepared["date"].dt.month
equipment_prepared["month_name"] = equipment_prepared["date"].dt.month_name()
equipment_prepared["quarter"] = equipment_prepared["date"].dt.quarter
equipment_prepared["day_of_week"] = equipment_prepared["date"].dt.day_name()
equipment_prepared["downtime_hours"] = (equipment_prepared["maintenance_hours"] + equipment_prepared["breakdown_hours"]).round(2)
equipment_prepared["has_breakdown"] = (equipment_prepared["breakdown_hours"] > 0).astype(int)
equipment_prepared["has_maintenance"] = (equipment_prepared["maintenance_hours"] > 0).astype(int)
equipment_prepared["maintenance_breakdown_overlap_flag"] = (
    (equipment_prepared["breakdown_hours"] > 0) & (equipment_prepared["maintenance_hours"] > 0)
).astype(int)
equipment_prepared["day_type"] = np.select(
    [equipment_prepared["has_breakdown"] == 1, equipment_prepared["has_maintenance"] == 1],
    ["Breakdown day", "Maintenance-only day"],
    default="Normal day",
)
equipment_prepared["hours_rounding_flag"] = (
    (equipment_prepared["operating_hours"] + equipment_prepared["idle_hours"] - equipment_prepared["available_hours"]).round(2) != 0
).astype(int)

added_equipment_columns = [c for c in equipment_prepared.columns if c not in source_equipment_columns]
equipment_prepared = equipment_prepared[source_equipment_columns + added_equipment_columns]

# ---- Safety
source_safety_columns = list(safety_raw.columns)

safety_prepared = safety_df.copy()
safety_prepared["month"] = safety_prepared["date"].dt.month
safety_prepared["month_name"] = safety_prepared["date"].dt.month_name()
safety_prepared["quarter"] = safety_prepared["date"].dt.quarter
safety_prepared["is_lost_time_injury"] = lti.astype(int)
safety_prepared["is_medical_treatment_injury"] = mti.astype(int)
safety_prepared["is_open_investigation"] = (safety_prepared["investigation_status"] != "Closed").astype(int)
category_map = {
    "Lost Time Injury (LTI)": "Injury",
    "Medical Treatment Injury (MTI)": "Injury",
    "First Aid Injury (FAI)": "Injury",
    "Near Miss": "Near miss or potential incident",
    "Potential Significant Incident (PSI)": "Near miss or potential incident",
    "Environmental Incident": "Environmental",
}
safety_prepared["incident_category"] = safety_prepared["incident_type"].map(category_map)
assert safety_prepared["incident_category"].notna().all(), "An incident type is not in the category map."

added_safety_columns = [c for c in safety_prepared.columns if c not in source_safety_columns]

print("=== EQUIPMENT ===")
print("Source rows:", len(equipment_raw), "| Prepared rows:", len(equipment_prepared))
print("Added columns:", added_equipment_columns)
print("Hours rounding flagged:", int(equipment_prepared["hours_rounding_flag"].sum()))
print("Maintenance and breakdown overlap flagged:", int(equipment_prepared["maintenance_breakdown_overlap_flag"].sum()))
print("\n=== SAFETY ===")
print("Source rows:", len(safety_raw), "| Prepared rows:", len(safety_prepared))
print("Added columns:", added_safety_columns)
print("Open investigations flagged:", int(safety_prepared["is_open_investigation"].sum()))

display(equipment_prepared.head(3))
display(safety_prepared.head(3))

=== EQUIPMENT ===
Source rows: 8030 | Prepared rows: 8030
Added columns: ['site_name', 'operation_mode', 'month', 'month_name', 'quarter', 'day_of_week', 'downtime_hours', 'has_breakdown', 'has_maintenance', 'maintenance_breakdown_overlap_flag', 'day_type', 'hours_rounding_flag']
Hours rounding flagged: 160
Maintenance and breakdown overlap flagged: 8

=== SAFETY ===
Source rows: 360 | Prepared rows: 360
Added columns: ['month', 'month_name', 'quarter', 'is_lost_time_injury', 'is_medical_treatment_injury', 'is_open_investigation', 'incident_category']
Open investigations flagged: 97


,date,site_id,equipment_id,equipment_type,model,autonomous,available_hours,operating_hours,idle_hours,maintenance_hours,breakdown_hours,availability_pct,utilization_pct,oee_pct,fuel_consumed_liters,maintenance_cost_aud,site_name,operation_mode,month,month_name,quarter,day_of_week,downtime_hours,has_breakdown,has_maintenance,maintenance_breakdown_overlap_flag,day_type,hours_rounding_flag
0,2025-01-01,IRON001,TRK-IR-001,Haul Truck,CAT 793F AHS,1,24.0,24.0,0.0,0.0,0.0,100.0,100.0,100.0,4320,1537.80,Iron Ridge Mine,Autonomous,1,January,1,Wednesday,0.0,0,0,0,Normal day,0
1,2025-01-02,IRON001,TRK-IR-001,Haul Truck,CAT 793F AHS,1,24.0,20.7,3.3,0.0,0.0,100.0,86.1,86.1,3718,1803.65,Iron Ridge Mine,Autonomous,1,January,1,Thursday,0.0,0,0,0,Normal day,0
2,2025-01-03,IRON001,TRK-IR-001,Haul Truck,CAT 793F AHS,1,24.0,21.1,2.9,0.0,0.0,100.0,88.1,88.1,3805,1197.03,Iron Ridge Mine,Autonomous,1,January,1,Friday,0.0,0,0,0,Normal day,0


,incident_id,date,site_id,site_name,incident_type,severity,description,days_lost,root_cause,investigation_status,workforce_size,month,month_name,quarter,is_lost_time_injury,is_medical_treatment_injury,is_open_investigation,incident_category
0,INC-00001,2025-02-27,IRON001,Iron Ridge Mine,Lost Time Injury (LTI),High,Fall from height,28,Procedural Gap,Closed,850,2,February,1,1,0,0,Injury
1,INC-00002,2025-11-24,IRON001,Iron Ridge Mine,Lost Time Injury (LTI),High,Crush injury,9,Human Error,Closed,850,11,November,4,1,0,0,Injury
2,INC-00003,2025-01-16,IRON001,Iron Ridge Mine,Medical Treatment Injury (MTI),Medium,Laceration,0,Training Gap,Closed,850,1,January,1,0,1,0,Injury


## 6. Export and Verification

Dataset siap analisis diekspor bersama log audit dan kamus data. Setiap file dibaca kembali dan dibandingkan dengan database.

In [15]:
descriptions = {
    # equipment source columns
    "date": "Calendar date of the record",
    "site_id": "Mine site identifier",
    "equipment_id": "Equipment unit identifier",
    "equipment_type": "Equipment type (Haul Truck, Excavator, Dozer, LHD Loader, Jumbo Drill)",
    "model": "Equipment model",
    "autonomous": "1 if the unit is autonomous, 0 if conventional",
    "available_hours": "Hours available in the day = 24 - maintenance - breakdown",
    "operating_hours": "Hours the unit operated",
    "idle_hours": "Hours available but idle (operating + idle = available, +/-0.1 rounding)",
    "maintenance_hours": "Planned maintenance hours",
    "breakdown_hours": "Breakdown hours",
    "availability_pct": "Available hours / 24 x 100",
    "utilization_pct": "Operating hours / available hours x 100",
    "oee_pct": "Availability x utilization / 100",
    "fuel_consumed_liters": "Fuel consumed (liters)",
    "maintenance_cost_aud": "Maintenance cost (AUD)",
    # equipment added
    "site_name": "Mine site name (joined from mine_sites)",
    "operation_mode": "Autonomous or Conventional (from autonomous)",
    "month": "Calendar month number",
    "month_name": "Calendar month name",
    "quarter": "Calendar quarter",
    "day_of_week": "Day of the week",
    "downtime_hours": "maintenance_hours + breakdown_hours",
    "has_breakdown": "1 if breakdown_hours > 0",
    "has_maintenance": "1 if maintenance_hours > 0",
    "maintenance_breakdown_overlap_flag": "1 if maintenance and breakdown both occurred on the day",
    "day_type": "Breakdown day, Maintenance-only day, or Normal day (breakdown takes precedence)",
    "hours_rounding_flag": "1 if operating + idle differs from available by a rounding step (0.1 h)",
    # safety source columns
    "incident_id": "Safety incident identifier",
    "incident_type": "Incident type (LTI, MTI, FAI, PSI, Near Miss, Environmental Incident)",
    "severity": "Low, Medium, or High (High = lost time injury)",
    "description": "Short description of the incident",
    "days_lost": "Working days lost (only on lost time injuries)",
    "root_cause": "Recorded root cause",
    "investigation_status": "Closed or Under Investigation",
    "workforce_size": "Site workforce count at the time of the incident",
    # safety added
    "is_lost_time_injury": "1 if the incident is a lost time injury (LTI)",
    "is_medical_treatment_injury": "1 if the incident is a medical treatment injury (MTI)",
    "is_open_investigation": "1 if investigation_status is not Closed",
    "incident_category": "Injury, Near miss or potential incident, or Environmental",
}

dictionary_rows = []
for dataset, frame, source_columns in [
    ("equipment_prepared.csv", equipment_prepared, source_equipment_columns),
    ("safety_prepared.csv", safety_prepared, source_safety_columns),
]:
    for column in frame.columns:
        dictionary_rows.append({
            "dataset": dataset,
            "column": column,
            "origin": "source" if column in source_columns else "added",
            "dtype": str(frame[column].dtype),
            "description": descriptions.get(column, ""),
        })
data_dictionary = pd.DataFrame(dictionary_rows)
missing_descriptions = data_dictionary[data_dictionary["description"] == ""]
assert missing_descriptions.empty, f"Columns without a description: {missing_descriptions['column'].tolist()}"

audit_df = pd.DataFrame(audit_log)

export_files = {
    "equipment_prepared.csv": equipment_prepared,
    "safety_prepared.csv": safety_prepared,
    "data_quality_audit.csv": audit_df,
    "prepared_data_dictionary.csv": data_dictionary,
}
for file_name, table in export_files.items():
    table.to_csv(OUTPUT_DIR / file_name, index=False)

print("Files written:", sorted(p.name for p in OUTPUT_DIR.iterdir()))

Files written: ['data_quality_audit.csv', 'equipment_prepared.csv', 'prepared_data_dictionary.csv', 'safety_prepared.csv']


In [16]:
# Read the files back and compare them with the database
exported = {name: pd.read_csv(OUTPUT_DIR / name) for name in export_files}
eq_back = exported["equipment_prepared.csv"]
sa_back = exported["safety_prepared.csv"]


def scalar(query):
    return float(pd.read_sql_query(query, conn).iloc[0, 0])


numeric_equipment = ["available_hours", "operating_hours", "idle_hours", "maintenance_hours", "breakdown_hours",
                     "availability_pct", "utilization_pct", "oee_pct", "fuel_consumed_liters", "maintenance_cost_aud"]

checks = [
    ("Equipment: rows", len(eq_back), scalar("SELECT COUNT(*) FROM equipment_performance")),
    ("Equipment: total breakdown hours", eq_back["breakdown_hours"].sum(), scalar("SELECT SUM(breakdown_hours) FROM equipment_performance")),
    ("Equipment: total maintenance hours", eq_back["maintenance_hours"].sum(), scalar("SELECT SUM(maintenance_hours) FROM equipment_performance")),
    ("Equipment: total maintenance cost (AUD)", eq_back["maintenance_cost_aud"].sum(), scalar("SELECT SUM(maintenance_cost_aud) FROM equipment_performance")),
    ("Equipment: total fuel (liters)", eq_back["fuel_consumed_liters"].sum(), scalar("SELECT SUM(fuel_consumed_liters) FROM equipment_performance")),
    ("Equipment: downtime hours = breakdown + maintenance", eq_back["downtime_hours"].sum(),
     scalar("SELECT SUM(breakdown_hours + maintenance_hours) FROM equipment_performance")),
    ("Safety: rows", len(sa_back), scalar("SELECT COUNT(*) FROM safety_incidents")),
    ("Safety: total days lost", sa_back["days_lost"].sum(), scalar("SELECT SUM(days_lost) FROM safety_incidents")),
    ("Safety: open investigations", sa_back["is_open_investigation"].sum(),
     scalar("SELECT COUNT(*) FROM safety_incidents WHERE investigation_status != 'Closed'")),
    ("Safety: lost time injuries", sa_back["is_lost_time_injury"].sum(),
     scalar("SELECT COUNT(*) FROM safety_incidents WHERE incident_type LIKE '%(LTI)%'")),
]
verification = pd.DataFrame(checks, columns=["check", "exported_file", "database"])
verification["match"] = np.isclose(verification["exported_file"], verification["database"], rtol=1e-9, atol=0.01)
display(verification)

# Source columns must be unchanged by the preparation
for name, back, raw, columns in [
    ("equipment", eq_back, equipment_raw, source_equipment_columns),
    ("safety", sa_back, safety_raw, source_safety_columns),
]:
    same = all(
        np.allclose(back[c].to_numpy(dtype=float), raw[c].to_numpy(dtype=float))
        if pd.api.types.is_numeric_dtype(raw[c])
        else (back[c].astype(str).to_numpy() == raw[c].astype(str).to_numpy()).all()
        for c in columns
    )
    print(f"{name}: source columns identical to the database -> {same}")
    assert same, f"{name}: a source column changed during preparation."

file_summary = pd.DataFrame([
    {"file": name, "rows": len(df), "columns": df.shape[1], "empty_cells": int(df.isna().sum().sum()),
     "size_kb": round((OUTPUT_DIR / name).stat().st_size / 1024, 1)}
    for name, df in exported.items()
])
display(file_summary)

assert verification["match"].all(), "An exported total does not match the database."
assert not eq_back.duplicated(["date", "equipment_id"]).any(), "Duplicate equipment keys in the export."
assert sa_back["incident_id"].is_unique, "Duplicate incident IDs in the export."
assert eq_back[added_equipment_columns].isna().sum().sum() == 0, "Missing values in added equipment columns."
assert sa_back[added_safety_columns].isna().sum().sum() == 0, "Missing values in added safety columns."
print("All checks passed.")

,check,exported_file,database,match
0,Equipment: rows,8030.00,8030.00,True
1,Equipment: total breakdown hours,1203.80,1203.80,True
2,Equipment: total maintenance hours,2013.10,2013.10,True
3,Equipment: total maintenance cost (AUD),21495749.22,21495749.22,True
4,Equipment: total fuel (liters),29772492.00,29772492.00,True
5,Equipment: downtime hours = breakdown + mainte...,3216.90,3216.90,True
6,Safety: rows,360.00,360.00,True
7,Safety: total days lost,111.00,111.00,True
8,Safety: open investigations,97.00,97.00,True
9,Safety: lost time injuries,8.00,8.00,True


equipment: source columns identical to the database -> True
safety: source columns identical to the database -> True


,file,rows,columns,empty_cells,size_kb
0,equipment_prepared.csv,8030,28,0,1395.0
1,safety_prepared.csv,360,18,0,57.0
2,data_quality_audit.csv,40,5,40,2.5
3,prepared_data_dictionary.csv,46,5,0,3.9


All checks passed.


**Interpretasi verifikasi**

Semua total pada file ekspor sama dengan database, kolom sumber pada kedua dataset identik dengan tabel sumber, tidak ada key ganda, dan kolom tambahan tidak memiliki nilai kosong. Notebook berhenti dengan pesan galat bila salah satunya tidak terpenuhi.

`data_quality_audit.csv` memuat seluruh aturan yang diperiksa di notebook ini beserta statusnya, dan `prepared_data_dictionary.csv` menjelaskan setiap kolom (kolom sumber dan kolom tambahan).

## 7. Summary

This notebook audited the equipment and safety tables of the synthetic mining operations database and produced two analysis-ready datasets.

### Audit findings

- **Equipment:** 22 units with 365 days each (8,030 records). No missing values, duplicates, negatives, or out-of-range values, and all 14 rules pass. Availability equals available hours ÷ 24, utilization equals operating hours ÷ available hours, and OEE equals availability × utilization. Available hours already exclude maintenance and breakdown, so operating + idle + maintenance + breakdown always equals 24 hours. 160 records (2.0%) differ by 0.1 hour from rounding, and 8 days have both maintenance and breakdown.
- **Safety:** 360 incidents, no missing values or duplicates, and all 15 safety rules pass. All 8 High-severity incidents are lost-time injuries and account for all 111 days lost. TRIFR and LTIFR match a recalculation from hours worked.
- **Open items:** 97 investigations (26.9%) are still open, including 2 lost-time injuries at Golden Valley (open about 285 and 47 days at year end). The open share is nearly equal across severities.
- **Coverage:** Nickel Point Mine (*Care & Maintenance*) has no equipment or safety records, consistent with its status.
- **Synthetic patterns to keep in mind:** every site has exactly 90 incidents with the same composition, and hours worked equal workforce × 4,380 hours. Safety rates therefore differ only because of workforce size.

### Corrections to the earlier draft of this notebook

- Availability was reported as inconsistent in 7,618 records. That came from testing the wrong formula, not from a data error.
- 651 records were flagged "hours exceed available". That also came from a wrong identity, because available hours already exclude maintenance and breakdown. The flag is removed.
- The equipment count was given as 21 units. The correct count is 22.

### Exported files (folder `prepared_data`)

1. `equipment_prepared.csv`: one row per unit per day, source columns unchanged plus 12 added columns.
2. `safety_prepared.csv`: one row per incident, source columns unchanged plus 7 added columns.
3. `data_quality_audit.csv`: every audit rule with its violation count and status.
4. `prepared_data_dictionary.csv`: all columns with origin and description.

All exported totals reconcile with the database.

**Data note:** The dataset is synthetic and used for portfolio and analytical demonstration. It does not represent actual operations of a mining company.